# 03 — Spatial Exposure and Equity Analysis

This notebook prepares the p[rational exposure variables used by the inspecton-priority model.

This script: 

- Derives an `arterial_streets` layer from `ARTDESCRIPT` in the Seattle Streets dataset.
- Calculates proximity to sidewalks, trails, schools, and arterial streets.
- Optionally joins Seattle's Racial and Social Equity Index polygons.
  - Presrves the RSE composite percentile, quantile, and raw score.
  - The script then uses the composite percentile to flag high-equity-priority tracts.

> Proximity variables represent potential exposure or operational consequence, but do not necessarily indicate that a tree is hazardous. 

In [ ]:
from pathlib import Path
import os
import pandas as pd
import arcpy

PROJECT_ROOT = Path(os.environ.get("SEATTLE_TREE_PROJECT_ROOT", Path.cwd())).resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
GDB = PROJECT_ROOT / "data" / "Seattle_Tree_Assets.gdb"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TREES_CLEAN = str(GDB / "trees_clean")
TREES_SPATIAL = str(GDB / "trees_spatial")
STREETS = str(GDB / "streets")
ARTERIAL_STREETS = str(GDB / "arterial_streets")
SIDEWALKS = str(GDB / "sidewalks")
TRAILS = str(GDB / "trails")
SCHOOLS = str(GDB / "schools")
RSE_POLYGONS = str(GDB / "rse_index")

if not arcpy.Exists(TREES_CLEAN):
    raise FileNotFoundError(f"Run revised Notebook 2 first. Missing: {TREES_CLEAN}")

if arcpy.Exists(TREES_SPATIAL):
    arcpy.management.Delete(TREES_SPATIAL)

arcpy.management.CopyFeatures(TREES_CLEAN, TREES_SPATIAL,)
print( f"Copied {int(arcpy.management.GetCount(TREES_SPATIAL)[0]):,} trees.")

# 1. Derive arterial streets

Seattle did not publish a "major roads" layer so this notebook derives a subset from the `ARTDESCRIPT` field in the Seattle Streets dataset. 

Classes used in the derived layer:
- Principal Arterial
- Minor Arterial
- Collector Arterial

Freeways were excluded because they represent a different exposure context from urban streets.

In [ ]:
ARTERIAL_CLASSES = ["Principal Arterial", "Minor Arterial", "Collector Arterial",]

if not arcpy.Exists(STREETS):
    raise FileNotFoundError(f"Seattle Streets feature class not found: {STREETS}")

street_fields = {f.name for f in arcpy.ListFields(STREETS)}

if "ARTDESCRIPT" not in street_fields:
    raise RuntimeError("ARTDESCRIPT was not found in the Seattle Streets layer.")

field_sql = arcpy.AddFieldDelimiters(STREETS, "ARTDESCRIPT",)
class_sql = ", ".join("'" + value.replace("'", "''") + "'" for value in ARTERIAL_CLASSES)
where_clause = (f"{field_sql} IN ({class_sql})")

if arcpy.Exists(ARTERIAL_STREETS):
    arcpy.management.Delete(ARTERIAL_STREETS)

arcpy.analysis.Select(in_features=STREETS, out_feature_class=ARTERIAL_STREETS, where_clause=where_clause,)

arterial_count = int(arcpy.management.GetCount(ARTERIAL_STREETS)[0])
print(f"Selected arterial segments: {arterial_count:,}")
print(f"Output: {ARTERIAL_STREETS}")

if arterial_count == 0:
    unique_values = sorted({
        row[0]
        for row in arcpy.da.SearchCursor(STREETS, ["ARTDESCRIPT"],)
        if row[0] not in (None, "")
    })
    print("Available ARTDESCRIPT values:")
    for value in unique_values:
        print(" -", value)
    raise RuntimeError(
        "No arterial segments were selected. Review ARTDESCRIPT values."
    )

# 2. Add spatial-analysis fields

In [ ]:
def add_field_if_missing(feature_class, field_name, field_type, length=None, alias=None,):
    existing = {field.name for field in arcpy.ListFields(feature_class)}
    if field_name not in existing:
        kwargs = {}
        if length: 
            kwargs["field_length"] = length
        if alias:
            kwargs["field_alias"] = alias
        arcpy.management.AddField(feature_class, field_name, field_type, **kwargs,)

EXPOSURE_LAYERS = {
    "SIDEWALK": {"fc": SIDEWALKS, "threshold_ft": 25, "required": True,},
    "TRAIL" : {"fc" : TRAILS, "threshold_ft": 40, "required": False,},
    "SCHOOL" : {"fc": SCHOOLS, "threshold_ft": 500, "required": False,},
    "ARTERIAL" : {"fc" : ARTERIAL_STREETS, "threshold_ft": 75, "required": False,},
}

for prefix in EXPOSURE_LAYERS:
    add_field_if_missing(TREES_SPATIAL, f"D_{prefix}_FT", "DOUBLE", alias=f"Distance to {prefix.title()} (feet)",)
    add_field_if_missing(TREES_SPATIAL, f"NEAR_{prefix}", "SHORT", alias=f"Near {prefix.title()}",)

add_field_if_missing(TREES_SPATIAL, "RSE_PERCENT", "DOUBLE", alias="Racial and Social Equity Percent",)
add_field_if_missing(TREES_SPATIAL, "RSE_QUINT", "Text", length=50, alias="Racial and Social Equity Quantile",)
add_field_if_missing(TREES_SPATIAL, "RSE_RAW", "DOUBLE",)
add_field_if_missing(TREES_SPATIAL, "EQUITY_HIGH", "SHORT", alias="High Equity Priority Area",)
add_field_if_missing(TREES_SPATIAL, "SPATIAL_NOTE", "TEXT", length=500, alias="Spatial Processing Notes",)

# 3. Calculate proximity indicators with Near and an update cursor

In [ ]:
def calculate_near_distance(target_fc, near_fc, distance_field, indicator_field, threshold_ft,):
    """
    Calculate nearest-feature distance and populate permanent distance and binary proximity fields.
    Uses an UpdateCursor instead of CalculateField to avoid Field Calculator expression errors.
    """
    print(f"  Running Near against: {near_fc}")
    arcpy.analysis.Near(in_features=target_fc, near_features=near_fc, method="PLANAR",)
    existing_fields = {field.name for field in arcpy.ListFields(target_fc)}
    
    if "NEAR_DIST" not in existing_fields:
        raise RuntimeError("Near completed without creating NEAR_DIST.")
        
    updated_count = 0
    no_near_feature_count = 0
    within_threshold_count = 0

    cursor_fields = ["NEAR_DIST", distance_field, indicator_field,]
    with arcpy.da.UpdateCursor(target_fc, cursor_fields, explicit=True,) as cursor:
        for row in cursor:
            near_distance = row[0]
            # Near uses -1 when no nearby feature is found.
            if (near_distance is None or float(near_distance) < 0):
                row[1] = None
                row[2] = 0
                no_near_feature_count += 1
            else:
                distance = float(near_distance)
                row[1] = distance
                row[2] = (1 if distance <= float(threshold_ft) else 0)

                if row[2] == 1:
                    within_threshold_count += 1

            cursor.updateRow(row)
            updated_count += 1

    removable_fields = [field_name for field_name in ["NEAR_FID", "NEAR_DIST", "NEAR_X", "NEAR_Y",  "NEAR_ANGLE",]
        if field_name in {field.name for field in arcpy.ListFields(target_fc)}]

    if removable_fields:
        arcpy.management.DeleteField(target_fc, removable_fields,)

    print(f"  Updated records:       {updated_count:,}")
    print(f"  Within threshold:      {within_threshold_count:,}")
    print(f"  No near feature found: {no_near_feature_count:,}")

In [ ]:
missing_layers = []

for prefix, settings in EXPOSURE_LAYERS.items():
    feature_class = settings["fc"]

    if arcpy.Exists(feature_class):
        print(f"Calculating proximity to "  f"{prefix.lower()}...")

        calculate_near_distance(
            target_fc=TREES_SPATIAL,
            near_fc=feature_class,
            distance_field=f"D_{prefix}_FT",
            indicator_field=f"NEAR_{prefix}",
            threshold_ft=settings["threshold_ft"],
        )

    else:
        message = (f"{prefix}: skipped because the " 
                   f"feature class was not found.")
        print(message)
        missing_layers.append(prefix)

        with arcpy.da.UpdateCursor(TREES_SPATIAL,[f"D_{prefix}_FT", f"NEAR_{prefix}",], explicit=True,) as cursor:
            for row in cursor:
                row[0] = None
                row[1] = 0
                cursor.updateRow(row)
        if settings["required"]:
            raise FileNotFoundError(f"Required exposure layer missing: " f"{feature_class}")

# 4. Configure and inspect RSE fields

In [ ]:
RSE_PERCENTILE_FIELD = "COMPOSITE_PERCENTILE"
RSE_QUINTILE_FIELD = "COMPOSITE_QUINTILE"
RSE_RAW_FIELD = "COMPOSITE_SCORE"

available_rse = {f.name for f in arcpy.ListFields(RSE_POLYGONS)}

for field_name in [RSE_PERCENTILE_FIELD, RSE_QUINTILE_FIELD, RSE_RAW_FIELD]:
    if field_name not in available_rse:
        raise RuntimeError(f"Missing RSE field: {field_name}")

percentile_values = [float(row[0]) for row in arcpy.da.SearchCursor(RSE_POLYGONS, [RSE_PERCENTILE_FIELD],)
                                                                   if row[0] is not None]
percentile_min = min(percentile_values)
percentile_max = max(percentile_values)

if percentile_max <= 1.0:
    RSE_HIGH_THRESHOLD = 0.75
    percentile_scale = "0–1"
else:
    RSE_HIGH_THRESHOLD = 75.0
    percentile_scale = "0–100"

print(f"Percentile range: {percentile_min} to {percentile_max}")
print(f"Detected scale: {percentile_scale}")
print(f"High threshold: {RSE_HIGH_THRESHOLD}")

# 5. Combine RSE values with tree assets via Spatial Join

In [ ]:
joined = str(GDB / "trees_rse_joined")

if arcpy.Exists(joined):
    arcpy.management.Delete(joined)

arcpy.analysis.SpatialJoin(
    target_features=TREES_SPATIAL, 
    join_features=RSE_POLYGONS,
    out_feature_class=joined,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    match_option="INTERSECT",
)

joined_fields = {f.name for f in arcpy.ListFields(joined)}

required_joined = [RSE_PERCENTILE_FIELD, RSE_QUINTILE_FIELD, RSE_RAW_FIELD, "RSE_PERCENT", "RSE_RAW", "EQUITY_HIGH"]

missing_joined = [f for f in required_joined if f not in joined_fields]
if missing_joined:
    raise RuntimeError(f"Missing joined fields: {missing_joined}")

In [ ]:
matched_count = unmatched_count = high_equity_count = 0

with arcpy.da.UpdateCursor(
    joined,
    [RSE_PERCENTILE_FIELD, RSE_QUINTILE_FIELD, RSE_RAW_FIELD, "RSE_PERCENT", "RSE_QUINT", "RSE_RAW", "EQUITY_HIGH",],explicit=True,) as cursor:
    for row in cursor:
        source_percentile = row[0]
        source_quintile = row[1]
        source_raw = row[2]

        row[4] = str(source_quintile) if source_quintile not in (None, "") else None
        row[5] = float(source_raw) if source_raw is not None else None

        if source_percentile is None:
            row[3] = None
            row[6] = 0
            unmatched_count += 1
        else:
            percentile = float(source_percentile)
            row[3] = percentile
            row[6] = 1 if percentile >= RSE_HIGH_THRESHOLD else 0
            matched_count += 1
            high_equity_count += int(row[6])

        cursor.updateRow(row)

print(f"Matched to RSE tract:       {matched_count:,}")
print(f"No RSE match:               {unmatched_count:,}")
print(f"High-equity-priority tract: {high_equity_count:,}")

arcpy.management.Delete(TREES_SPATIAL)
arcpy.management.Rename(joined, Path(TREES_SPATIAL).name)

# 6. Summaries and validation

In [ ]:
total_trees = int(arcpy.management.GetCount(TREES_SPATIAL)[0])

summary_records = []

for prefix, settings in EXPOSURE_LAYERS.items():
    within_count = sum(int(row[0] or 0) 
                      for row in arcpy.da.SearchCursor(TREES_SPATIAL, [f"NEAR_{prefix}"],))

    summary_records.append({
        "criterion" : prefix.title(),
        "threshold_ft" : settings["threshold_ft"],
        "tree_count" : within_count,
        "share_pct" : round(within_count / total_trees * 100, 2),
    })

exposure_summary = pd.DataFrame(summary_records)
exposure_summary.to_csv(OUTPUT_DIR / "spatial_exposure_summary.csv", index=False,)

display(exposure_summary)

equity_summary = pd.DataFrame([
    {"metric" : "Matched to RSE tract", 
    "tree_count": matched_count,
    "share_pct": round(matched_count / total_trees * 100, 2),},
    
    {"metric" : "High-equity-priority tract",
    "tree_count" : high_equity_count,
    "share_pct": round(high_equity_count / total_trees * 100, 2),},

    {"metric" : "No RSE match",
    "tree_count" : unmatched_count,
    "share_pct" : round(unmatched_count / total_trees * 100, 2),},
])

equity_summary.to_csv(OUTPUT_DIR / "rse_equity_summary.csv",
                     index=False,)
display(equity_summary)

In [ ]:
note_parts = []

if missing_layers:
    note_parts.append("Missing optional sources: " + ", ".join(sorted(set(missing_layers))))

note_parts.append(f"RSE threshold {RSE_HIGH_THRESHOLD} on {percentile_scale} percentile_scale")

note = "; ".join(note_parts)

with arcpy.da.UpdateCursor(TREES_SPATIAL, ["SPATIAL_NOTE"], explicit=True) as cursor:
    for row in cursor:
        row[0] = note
        cursor.updateRow(row)

required_final_fields = ["D_SIDEWALK_FT", "NEAR_SIDEWALK", 
                         "D_TRAIL_FT", "NEAR_TRAIL", 
                         "D_SCHOOL_FT", "NEAR_SCHOOL",
                         "D_ARTERIAL_FT", "NEAR_ARTERIAL", 
                         "RSE_PERCENT", "RSE_QUINT", 
                         "RSE_RAW", "EQUITY_HIGH",]

available_final = {f.name for f in arcpy.ListFields(TREES_SPATIAL)}
missing_final = [f for f in required_final_fields if f not in available_final]

if missing_final:
    raise RuntimeError(f"Missing final fields: {missing_final}")

print(f"Final tree count: {total_trees:,}")
print(f"Output: {TREES_SPATIAL}")
print("Notebook 3 completed successfully.")